# CME538 - Introduction to Data Science

## Assignment 5 - Geospatial Analysis

### Learning Objectives

After completing this assignment, you should be able to:

- Load and inspect geospatial datasets using GeoPandas.
- Recognize and work with point, line, and polygon geometries.
- Create GeoDataFrames from longitude and latitude coordinates.
- Inspect and transform Coordinate Reference Systems (CRS).
- Use a projected coordinate system for distance and area calculations.
- Create static geospatial visualizations using GeoPandas and Matplotlib.
- Calculate and compare spatial measures such as area and density.
- Create interactive maps using Folium.
- Perform spatial joins based on geographic relationships.
- Use buffers and spatial relationships for proximity analysis.
- Interpret geospatial results while recognizing the importance of CRS and spatial context.
- Write clear, concise, and reproducible geospatial analysis code.

Sample figures showing the expected general appearance of selected maps are available in the `images` folder. Your figures do not need to match these examples exactly.

### Marking Breakdown

| Question | Marks |
|---|---:|
| Question 1a | 1 |
| Question 1b | 1 |
| Question 2 | 1 |
| Question 3 | 1 |
| Question 4 | 1 |
| Question 5a | 1 |
| Question 5b | 2 |
| Question 5c | 1 |
| Question 6 | 2 |
| Question 7 | 2 |
| Question 8a | 1 |
| Question 8b | 1 |
| Question 8c | 2 |
| Question 8d | 1 |
| Code quality | 3 |
| **Total** | **20** |

### Code Quality

Code quality will be assessed across the complete notebook.

| Level | Points | Description |
|---|---:|---|
| **Developing** | 1 | Code produces the required results but may be difficult to follow, unnecessarily repetitive, poorly organized, or include excessive output. |
| **Competent** | 2 | Code is organized and readable, uses appropriate Python, Pandas, and GeoPandas operations, and produces concise, relevant outputs. |
| **Strong** | 3 | Code is clear, concise, well organized, and reproducible; uses Python and GeoPandas effectively; avoids unnecessary operations and output; and executes successfully from beginning to end. |

## Notebook Setup

In [ ]:
# Import required libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import requests

%matplotlib inline

# Overview

This assignment continues the City of Toronto Bike Share analysis from Assignment 4.

In the previous assignment, you investigated when people use Bike Share Toronto and how ridership varies with user type and weather. In this assignment, you will add a new dimension to the analysis: **location**.

You will use geospatial data to examine where Bike Share infrastructure is located, how station access varies across Toronto neighbourhoods, and how well Bike Share connects with other transportation infrastructure.

Throughout the assignment, you will work with point, line, and polygon data using GeoPandas and create both static and interactive maps.

---

# 1. Preparing Bike Share Station Data

We will begin by retrieving current information about **Bike Share Toronto stations** from the public station feed.

The station data include:

- station ID,
- station name,
- latitude,
- longitude, and
- station capacity.

At this stage, the station data will be stored in a standard Pandas DataFrame. Later, you will use the latitude and longitude coordinates to create a GeoPandas GeoDataFrame representing each station as a geographic point.

A backup file, `bike_share_stations.csv`, is also included in the assignment directory. If the online data source is unavailable, use the fallback code provided below.

In [ ]:
# Retrieve current Bike Share Toronto station information
url = "https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"

# Send a request to the station information feed
response = requests.get(url, timeout=10)
response.raise_for_status()

# Extract the station records from the JSON response
station_data = response.json()["data"]["stations"]

# Create a DataFrame containing the variables needed for this assignment
bikeshare_stations = (
    pd.DataFrame(station_data)
    [["station_id", "name", "lat", "lon", "capacity"]]
    .rename(columns={
        "station_id": "Station Id",
        "name": "Station Name"
    })
)

# Convert the station identifier to an integer
bikeshare_stations["Station Id"] = (
    bikeshare_stations["Station Id"].astype(int)
)

# Preview the station data
bikeshare_stations.head()

In [ ]:
# Fallback if the online data source is unavailable:
# bikeshare_stations = pd.read_csv("bike_share_stations.csv")

---

# 2. Toronto Neighbourhood Boundaries

Geospatial datasets often represent geographic features using geometric objects such as **points, lines, and polygons**.

The assignment directory contains `toronto_neighbourhoods.shp`, a shapefile containing the boundaries of Toronto neighbourhoods.

## Question 1 - Loading and Preparing Neighbourhood Data

### Question 1a - Load the Neighbourhood Boundaries

Use `gpd.read_file()` to load `toronto_neighbourhoods.shp` into a GeoDataFrame named `neighbourhoods`.

A GeoDataFrame is similar to a Pandas DataFrame, but it contains a special `geometry` column that stores the spatial representation of each feature.

In [ ]:
# Question 1a

# Load the Toronto neighbourhood boundaries using GeoPandas.
neighbourhoods = ...

# Preview the GeoDataFrame
neighbourhoods.head()

In [ ]:
# Verification - do not modify
print(f"Q1a Answer - Object type: {type(neighbourhoods).__name__}")
print(f"Number of neighbourhoods: {len(neighbourhoods)}")
print(f"Geometry column: {neighbourhoods.geometry.name}")
print(f"CRS: {neighbourhoods.crs}")

---

### Question 1b - Prepare the Neighbourhood Data

The neighbourhood dataset contains several columns that are not required for this analysis.

Prepare `neighbourhoods` by:

1. keeping only `FIELD_8` and `geometry`,
2. renaming `FIELD_8` to `name`, and
3. removing the neighbourhood ID shown in parentheses at the end of each name.

For example:

`Yorkdale-Glen Park (31)` → `Yorkdale-Glen Park`

In [ ]:
# Question 1b

# Keep only the neighbourhood-name and geometry columns.
neighbourhoods = ...

# Rename "FIELD_8" to "name".
neighbourhoods = ...

# Remove the neighbourhood ID in parentheses from the end of each name.
# Hint: str.replace() with a regular expression may be useful.
neighbourhoods["name"] = ...

# Preview the cleaned GeoDataFrame
neighbourhoods.head()

In [ ]:
# Verification - do not modify
print(f"Q1b Answer - Columns: {neighbourhoods.columns.tolist()}")
print(f"Missing neighbourhood names: {neighbourhoods['name'].isna().sum()}")
print(f"Names still ending in an ID: {neighbourhoods['name'].str.contains(r'\(\d+\)$', regex=True).sum()}")

---

### Geometry Types

The `geometry` column of a GeoDataFrame can contain different types of geometric objects. Three common types are **Point**, **LineString**, and **Polygon**.

<br>
<img src="images/shapes.png" alt="Examples of Point, LineString, and Polygon geometries" width="450"/>
<br>

Neighbourhood boundaries represent geographic areas, so we expect this dataset to contain polygon geometries.

Let's inspect the geometry types stored in `neighbourhoods`.

In [ ]:
# Inspect the geometry types in the neighbourhood dataset
neighbourhoods.geometry.geom_type.value_counts()

All 140 neighbourhood features are represented as `Polygon` geometries. This is appropriate because each geometry represents the boundary and area of a Toronto neighbourhood.

---

# 3. Toronto Bikeway Network

The City of Toronto maintains a geospatial dataset describing the city's bikeway network.

The assignment directory contains `bikeway_network.shp`. We will load the dataset using GeoPandas and prepare it for later analysis and visualization.

In [ ]:
# Load the Toronto bikeway network
bike_lanes = gpd.read_file("bikeway_network.shp")

# Preview the GeoDataFrame
bike_lanes.head()

Like the neighbourhood dataset, `bike_lanes` contains a `geometry` column. However, bikeways represent linear features rather than geographic areas.

Let's inspect the geometry types in the dataset.

In [ ]:
# Inspect the geometry types in the bikeway dataset
bike_lanes.geometry.geom_type.value_counts()

Most bikeway features are represented as `LineString` geometries, with a small number represented as `MultiLineString` geometries.

Both geometry types are appropriate for representing linear transportation infrastructure. A `LineString` represents a single connected line, while a `MultiLineString` can represent multiple line segments as one feature.

The dataset contains many attributes that are not required for this assignment. We will keep the route name, route type, length, and geometry, and rename the columns used in our analysis.

In [ ]:
# Keep only the columns required for the analysis
bike_lanes = bike_lanes[
    ["LF_NAME", "SEG_TYPE", "length", "geometry"]
].copy()

# Rename columns for clarity
bike_lanes = bike_lanes.rename(
    columns={
        "LF_NAME": "name",
        "SEG_TYPE": "route_type"
    }
)

# Preview the cleaned GeoDataFrame
bike_lanes.head()

The bikeway network contains several types of cycling infrastructure. Let's examine the route types represented in the dataset.

In [ ]:
# Examine the different route types
bike_lanes["route_type"].value_counts()

For this assignment, we will focus specifically on segments classified as `bike lane`.

In [ ]:
# Keep only segments classified as bike lanes
bike_lanes = (
    bike_lanes[
        bike_lanes["route_type"] == "bike lane"
    ]
    .copy()
)

# Preview the filtered GeoDataFrame
bike_lanes.head()

In [ ]:
# Check the filtered bikeway data
print(f"Number of bike lane segments: {len(bike_lanes)}")

print("\nGeometry types:")
print(bike_lanes.geometry.geom_type.value_counts())

After filtering, the dataset contains 3,574 bike lane features. Almost all are represented as `LineString` geometries, with one `MultiLineString` feature.

Both geometry types represent linear spatial features and can be used directly in the analyses that follow.

---

# 4. Converting Bike Share Stations to a GeoDataFrame

The Bike Share station data are currently stored in a standard Pandas DataFrame.

Each station includes a latitude and longitude, but these coordinates are not yet represented as spatial geometry.

In [ ]:
bikeshare_stations.head()

---

## Question 2 - Create a Station GeoDataFrame

Convert `bikeshare_stations` into a GeoDataFrame named `bikeshare_stations_gdf`.

Use `gpd.points_from_xy()` to create a `Point` geometry for each station from its longitude and latitude coordinates.

Remember that geographic coordinates are specified as:

`(x, y) = (longitude, latitude)`

The station coordinates use the **WGS 84** coordinate reference system, `EPSG:4326`. Assign this CRS when creating the GeoDataFrame.

In [ ]:
# Question 2

# Create Point geometries from longitude and latitude.
# Remember: x = longitude and y = latitude.
station_geometry = ...

# Convert bikeshare_stations to a GeoDataFrame.
# Assign the WGS 84 CRS: EPSG:4326.
bikeshare_stations_gdf = ...

bikeshare_stations_gdf.head()

In [ ]:
# Verification - do not modify
print(f"Q2 Answer - Object type: {type(bikeshare_stations_gdf).__name__}")
print(f"CRS: {bikeshare_stations_gdf.crs}")

print("\nGeometry types:")
print(bikeshare_stations_gdf.geometry.geom_type.value_counts())

---

# 5. A First Spatial Overlay

GeoPandas makes it easy to create quick spatial visualizations using the `.plot()` method.

Let's begin by plotting the Toronto neighbourhood boundaries.

In [ ]:
# Plot Toronto neighbourhood boundaries
neighbourhoods.plot(
    figsize=(10, 8),
    edgecolor="white",
    alpha=0.75
)

plt.title("Toronto Neighbourhoods")
plt.show()

Now let's try plotting all three spatial datasets on the same axes:

- Toronto neighbourhoods,
- Bike Share stations, and
- bike lanes.

Because all three datasets describe locations in Toronto, we might expect them to align.

In [ ]:
# Attempt to overlay the three spatial datasets
ax = neighbourhoods.plot(
    figsize=(10, 8),
    edgecolor="white",
    alpha=0.75
)

bikeshare_stations_gdf.plot(
    ax=ax,
    markersize=10
)

bike_lanes.plot(
    ax=ax,
    linewidth=1
)

plt.title("Spatial Layers Before CRS Alignment")
plt.show()

Something is wrong with the resulting visualization. The three datasets do not appear together as expected.

A useful first diagnostic is to compare their coordinate ranges and **Coordinate Reference Systems (CRS)**.

In [ ]:
# Compare the coordinate ranges of the three datasets
print("Neighbourhood bounds:")
print(neighbourhoods.total_bounds)

print("\nBike Share station bounds:")
print(bikeshare_stations_gdf.total_bounds)

print("\nBike lane bounds:")
print(bike_lanes.total_bounds)

# Compare their coordinate reference systems
print("\nCoordinate Reference Systems:")
print("Neighbourhoods:", neighbourhoods.crs)
print("Bike Share stations:", bikeshare_stations_gdf.crs)
print("Bike lanes:", bike_lanes.crs)

The neighbourhood and Bike Share station datasets both use `EPSG:4326`, where locations are represented using longitude and latitude.

The bike lane dataset uses `EPSG:3857`, a projected coordinate system with coordinates represented in metres.

This explains why the layers do not align in our first plot. Before spatial datasets can be meaningfully compared or combined, their coordinate reference systems must be compatible.

In the next section, we will examine coordinate reference systems more closely and transform the datasets to a common CRS.

---

# 6. Coordinate Reference Systems

A **Coordinate Reference System (CRS)** defines how coordinates in a spatial dataset correspond to locations on Earth.

Different CRS can represent the same geographic locations using very different coordinate values. This is why the layers in the previous section did not align.

<br>
<img src="images/crs.png" alt="Examples of different map projections" width="700"/>
<br>

GeoPandas stores CRS information with each GeoDataFrame. We can inspect it using the `.crs` attribute.

In [ ]:
# Compare the CRS of the three spatial datasets
print("Neighbourhoods:", neighbourhoods.crs)
print("Bike Share stations:", bikeshare_stations_gdf.crs)
print("Bike lanes:", bike_lanes.crs)

The neighbourhood and Bike Share station datasets use `EPSG:4326`, while the bike lane dataset uses `EPSG:3857`.

### EPSG:4326 — WGS 84

`EPSG:4326` is a geographic coordinate system commonly used for longitude and latitude.

Coordinates are expressed in **degrees**, making this CRS useful for storing geographic locations and displaying data on many web-based maps.

### EPSG:3857 — Web Mercator

`EPSG:3857` is a projected coordinate system widely used by online mapping platforms.

Unlike `EPSG:4326`, its coordinates are represented using projected x and y values rather than longitude and latitude.

The important point is that spatial datasets must generally use a **common CRS** before they are overlaid or used together in spatial operations.

GeoPandas uses `.to_crs()` to transform geometries from one coordinate reference system to another.

For example, we can transform the bike lanes from `EPSG:3857` to the geographic CRS used by the other datasets.

In [ ]:
# Reproject bike lanes to WGS 84
bike_lanes_wgs84 = bike_lanes.to_crs("EPSG:4326")

# Confirm the transformed CRS
bike_lanes_wgs84.crs

## Choosing a CRS for Spatial Analysis

The appropriate CRS depends on the analysis being performed.

Longitude and latitude are useful for geographic locations and web maps, but calculations such as **distance, area, and buffering** are generally easier to interpret in a suitable projected CRS whose units are metres.

Because our analysis is focused on the Toronto area, we will use **NAD83 / UTM Zone 17N (`EPSG:26917`)**.

This CRS uses metres and is appropriate for the local distance and area calculations used later in the assignment.

---

## Question 3 - Reprojecting the Spatial Data

Transform the following GeoDataFrames to `EPSG:26917`:

- `neighbourhoods`
- `bikeshare_stations_gdf`
- `bike_lanes`

Replace each variable with its reprojected version.

In [ ]:
# Question 3

# Reproject all three GeoDataFrames to NAD83 / UTM Zone 17N.
# Use EPSG:26917.

neighbourhoods = ...
bikeshare_stations_gdf = ...
bike_lanes = ...

In [ ]:
# Verification - do not modify
print("Q3 Answer - Coordinate Reference Systems:")
print("Neighbourhoods:", neighbourhoods.crs)
print("Bike Share stations:", bikeshare_stations_gdf.crs)
print("Bike lanes:", bike_lanes.crs)

---

## Question 4 - Plotting the Spatial Layers

Now that all three datasets use the same projected CRS, create a single static map containing:

- Toronto neighbourhood boundaries,
- Bike Share station locations, and
- bike lanes.

Include:

- a legend identifying Bike Share stations and bike lanes,
- an x-axis labelled `Easting (m)`,
- a y-axis labelled `Northing (m)`, and
- an appropriate title.

A sample figure is available in the `images` folder for reference.

<br>
<img src="images/q4.png" alt="Example spatial overlay of Toronto neighbourhoods, Bike Share stations, and bike lanes" width="500"/>
<br>

In [ ]:
# Question 4

# Create one figure and axes for all spatial layers.
fig, ax = plt.subplots(figsize=(10, 8))

# Plot neighbourhood boundaries.
# Hint: use facecolor="none" so the other layers remain visible.
...

# Plot bike lanes on the same axes.
...

# Plot Bike Share stations on the same axes.
...

# Add the required labels, title, and legend.
ax.set_xlabel(...)
ax.set_ylabel(...)
ax.set_title(...)
ax.legend()

plt.tight_layout()
plt.show()

---

# 7. Working with Geometry Properties

GeoPandas provides geometric properties and methods through the `geometry` column.

Because our GeoDataFrames now use the projected CRS `EPSG:26917`, coordinate values and measurements are expressed in **metres**.

Different geometry types provide different useful properties:

- **Point** — x and y coordinates
- **LineString / MultiLineString** — length
- **Polygon** — area

### Point Coordinates

For point geometries, `.geometry.x` and `.geometry.y` return the projected x and y coordinates.

In [ ]:
# Examine the coordinates of the first few Bike Share stations
bikeshare_stations_gdf[["Station Name"]].assign(
    x=bikeshare_stations_gdf.geometry.x,
    y=bikeshare_stations_gdf.geometry.y
).head()

### Line Length

For line geometries, `.geometry.length` calculates the length of each feature.

Because the current CRS uses metres, the resulting lengths are measured in metres.

In [ ]:
# Examine the length of the first few bike lane segments
bike_lanes.geometry.length.head()

### Polygon Area

For polygon geometries, `.geometry.area` calculates the area of each feature.

Because the current CRS uses metres, the resulting areas are measured in square metres.

In [ ]:
# Examine the area of the first few neighbourhoods
neighbourhoods.geometry.area.head()

---

# 8. Neighbourhood-Level Bike Share Analysis

The Bike Share network is not distributed evenly across Toronto. Some neighbourhoods contain many stations, while others contain relatively few.

A simple map of station locations can show the overall spatial pattern, but it can be difficult to compare neighbourhoods directly when many points overlap.

Let's first visualize the Bike Share stations together with the neighbourhood boundaries.

In [ ]:
# Plot neighbourhood boundaries and Bike Share station locations
fig, ax = plt.subplots(figsize=(10, 8))

neighbourhoods.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.5
)

bikeshare_stations_gdf.plot(
    ax=ax,
    markersize=10,
    label="Bike Stations"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Bike Share Stations Across Toronto Neighbourhoods")
ax.legend()

plt.tight_layout()
plt.show()

Although the map shows where stations are concentrated, comparing neighbourhoods using point locations alone is difficult.

We will therefore calculate two neighbourhood-level measures:

1. the **number of Bike Share stations**, and
2. the **number of stations per square kilometre**.

Station density accounts for differences in neighbourhood area and provides a more meaningful spatial comparison than station counts alone.

---

## Question 5a - Calculate Neighbourhood Area

Calculate the area of each neighbourhood in **square kilometres** and store the result in a new column named `area`.

Remember that `EPSG:26917` uses metres, so `.geometry.area` returns square metres.

In [ ]:
# Question 5a

# Calculate neighbourhood area in square kilometres.
# EPSG:26917 gives area in m², so convert m² to km².
neighbourhoods["area"] = ...

neighbourhoods.head()

In [ ]:
# Verification - do not modify
print("Q5a Answer - Area summary (km²):")
print(neighbourhoods["area"].describe().round(2))

---

## Question 5b - Count Bike Share Stations by Neighbourhood

Determine how many Bike Share stations are located within each Toronto neighbourhood.

Use the .within() spatial relationship to determine how many Bike Share stations fall inside each neighbourhood.

Then calculate the number of stations in each neighbourhood and store the result in a new column named `stations`.

Neighbourhoods containing no Bike Share stations should have a value of `0`.

Finally, sort `neighbourhoods` by `stations` in descending order.

A sample of the expected output is available in the `images` folder.

<br>
<img src="images/station_count.png" alt="Example neighbourhood Bike Share station counts" width="500"/>
<br>

In [ ]:
# Question 5b

# Count the number of Bike Share stations inside each neighbourhood.
# Hint: for each neighbourhood geometry, check which station points
# are within that polygon and count the True values.
neighbourhoods["stations"] = neighbourhoods.apply(
    lambda row: ...,
    axis=1
)

# Sort from highest to lowest station count.
neighbourhoods = ...

neighbourhoods.head(10)

In [ ]:
# Verification - do not modify
print(f"Q5b Answer - Total stations assigned: {neighbourhoods['stations'].sum()}")
print(f"Neighbourhoods with stations: {(neighbourhoods['stations'] > 0).sum()}")

print("\nTop five neighbourhoods:")
print(
    neighbourhoods[
        ["name", "stations"]
    ].head(5).to_string(index=False)
)

---

## Question 5c - Calculate Station Density

Station counts alone can be misleading because Toronto neighbourhoods have different geographic areas.

Calculate the **Bike Share station density** for each neighbourhood, measured as:

$$
\text{Station Density} =
\frac{\text{Number of Bike Share Stations}}
{\text{Neighbourhood Area (km}^2\text{)}}
$$

Store the result in a new column named `station_density`.

Then sort the neighbourhoods by `station_density` in descending order and display the 10 neighbourhoods with the highest station density.

In [ ]:
# Question 5c

# Calculate Bike Share stations per square kilometre.
neighbourhoods["station_density"] = ...

# Sort from highest to lowest station density.
neighbourhoods = ...

# Display the 10 neighbourhoods with the highest station density.
neighbourhoods[
    ["name", "area", "stations", "station_density"]
].head(10)

In [ ]:
# Verification - do not modify
print("Q5c Answer - Highest station densities:")
print(
    neighbourhoods[
        ["name", "station_density"]
    ]
    .head(5)
    .round(2)
    .to_string(index=False)
)

---

# 9. Interactive Maps with Folium

Static maps are useful for analysis and reporting, but interactive maps make it easier to explore spatial patterns.

In this section, we will use **Folium** to visualize Bike Share stations and neighbourhood-level station density.

Folium uses web-map coordinates in **latitude and longitude**. Our GeoDataFrames are currently projected to `EPSG:26917` for distance and area calculations, so we first create geographic versions in `EPSG:4326`.

In [ ]:
# Create geographic copies for interactive mapping
stations_wgs84 = bikeshare_stations_gdf.to_crs(epsg=4326)
neighbourhoods_wgs84 = neighbourhoods.to_crs(epsg=4326)

# Approximate centre of Toronto
toronto_center = [43.70, -79.40]

## 9.1 Marker Clusters

Displaying hundreds of individual station markers can make an interactive map difficult to read.

A `MarkerCluster` groups nearby stations together and separates them as the user zooms in.

In [ ]:
from folium.plugins import MarkerCluster

# Create the base map
station_map = folium.Map(
    location=toronto_center,
    tiles="CartoDB positron",
    zoom_start=10
)

# Create a marker cluster
station_cluster = MarkerCluster(
    name="Bike Share Stations"
).add_to(station_map)

# Add each Bike Share station
for _, row in stations_wgs84.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row["Station Name"]
    ).add_to(station_cluster)

station_map

## 9.2 Station Heat Map

Another way to visualize the spatial concentration of Bike Share stations is with a heat map.

Areas with a greater concentration of station locations will appear more intense on the map.

Note that this represents the **spatial concentration of station locations**, not ridership or neighbourhood station density.

In [ ]:
from folium.plugins import HeatMap

# Create the base map
heat_map = folium.Map(
    location=toronto_center,
    tiles="CartoDB positron",
    zoom_start=10
)

# Prepare station coordinates as [latitude, longitude]
station_locations = [
    [point.y, point.x]
    for point in stations_wgs84.geometry
]

# Add the heat map
HeatMap(
    station_locations,
    radius=15
).add_to(heat_map)

heat_map

## 9.3 Neighbourhood Station Density

A **choropleth map** represents a numerical variable using different colours across geographic regions.

We will use a choropleth to visualize the Bike Share station density calculated in Question 5.

Each neighbourhood will be coloured according to its number of Bike Share stations per square kilometre.

In [ ]:
# Create a copy for mapping
choropleth_data = neighbourhoods_wgs84[
    ["name", "station_density", "geometry"]
].copy()

# Create the base map
density_map = folium.Map(
    location=toronto_center,
    tiles="CartoDB positron",
    zoom_start=10
)

# Add neighbourhood station density
folium.Choropleth(
    geo_data=choropleth_data,
    data=choropleth_data,
    columns=["name", "station_density"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Bike Share Stations per km²"
).add_to(density_map)

# Add neighbourhood names and station densities as tooltips
folium.GeoJson(
    choropleth_data,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "transparent"
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "station_density"],
        aliases=["Neighbourhood:", "Stations per km²:"],
        localize=True
    )
).add_to(density_map)

density_map

---

## Question 6 - Mapping Bike Share and Weather Stations

In Assignment 4, you analyzed Bike Share ridership together with weather observations from the **Toronto City Centre** weather station. However, we did not examine where that weather station is located relative to the Bike Share network.

Create an interactive Folium map showing:

- Bike Share station locations using `CircleMarker()` objects, and
- the Toronto City Centre weather station at `(43.63, -79.40)` using a `Marker()` object.

Your map should:

- be centred on Toronto,
- show all Bike Share stations,
- clearly distinguish the weather station from the Bike Share stations, and
- include a tooltip identifying the weather station.

Remember that Folium expects locations in `(latitude, longitude)` coordinates, so use the `EPSG:4326` version of the Bike Share station data.

A sample map is available in the `images` folder for reference.

<br>
<img src="images/q6.png" alt="Example map showing Bike Share stations and the Toronto City Centre weather station" width="700"/>
<br>

In [ ]:
# Question 6

# Create a Folium map centred on Toronto.
weather_station_map = ...

# Add each Bike Share station using CircleMarker().
# stations_wgs84 has already been created earlier in the notebook.
for _, row in stations_wgs84.iterrows():
    ...

# Add the Toronto City Centre weather station at
# latitude 43.63, longitude -79.40 using Marker().
...

# Display the completed map.
weather_station_map

---

# 10. Spatial Joins

A **spatial join** combines GeoDataFrames based on the spatial relationship between their geometries rather than matching values in ordinary columns.

For example, a spatial join can determine:

- which neighbourhood contains a Bike Share station,
- which features intersect, or
- which locations fall within a geographic region.

GeoPandas provides `gpd.sjoin()` for performing spatial joins.

---

## Question 7 - Assigning Stations to Neighbourhoods

Determine which Toronto neighbourhood contains each Bike Share station.

Use `gpd.sjoin()` to spatially join `bikeshare_stations_gdf` with `neighbourhoods` using the spatial relationship `predicate="within"`.

Add the resulting neighbourhood name to `bikeshare_stations_gdf` as a new column named `neighbourhood`.

Before performing the spatial join, make sure both GeoDataFrames use the same CRS.

In [ ]:
# Question 7

# Spatially join Bike Share stations to neighbourhoods.
# Keep the neighbourhood "name" and "geometry" columns.
stations_with_neighbourhood = gpd.sjoin(
    ...,
    ...,
    how="left",
    predicate="within"
)

# Add the neighbourhood name to bikeshare_stations_gdf
# using a new column called "neighbourhood".
bikeshare_stations_gdf["neighbourhood"] = ...

bikeshare_stations_gdf.head()

In [ ]:
# Verification - do not modify
print(
    "Q7 Answer - Stations assigned to a neighbourhood:",
    bikeshare_stations_gdf["neighbourhood"].notna().sum()
)

print(
    "Stations without a neighbourhood:",
    bikeshare_stations_gdf["neighbourhood"].isna().sum()
)

print(
    "Unique neighbourhoods represented:",
    bikeshare_stations_gdf["neighbourhood"].nunique()
)

---

# 11. Proximity Analysis

Our final task is to examine how well Bike Share connects with the TTC subway network.

We will determine which subway stations are located within **200 metres** of at least one Bike Share station.

Because distance calculations require meaningful spatial units, all data used in this section should use the projected CRS `EPSG:26917`.

---

## Question 8a - Prepare TTC Subway Stations

Load `subway_stations.shp` as a GeoDataFrame named `subway_stations`.

Reproject the GeoDataFrame to `EPSG:26917`.

---

In [ ]:
# Question 8a

# Load the TTC subway station shapefile.
subway_stations = ...

# Reproject the subway stations to EPSG:26917.
subway_stations = ...

subway_stations.head()

In [ ]:
# Verification - do not modify
print(f"Q8a Answer - Number of subway stations: {len(subway_stations)}")
print(f"CRS: {subway_stations.crs}")
print("\nGeometry types:")
print(subway_stations.geometry.geom_type.value_counts())

---

## Question 8b - Create 200 Metre Bike Share Buffers

Create a 200 metre buffer around every Bike Share station.

Store the resulting geometries in a variable named `bikeshare_stations_buffer`.

Because `bikeshare_stations_gdf` uses `EPSG:26917`, the buffer distance is interpreted in metres.

In [ ]:
# Question 8b

# Create a 200 metre buffer around every Bike Share station.
# Because the current CRS uses metres, the buffer distance is 200.
bikeshare_stations_buffer = ...

bikeshare_stations_buffer.head()

In [ ]:
# Verification - do not modify
print(f"Q8b Answer - Number of buffers: {len(bikeshare_stations_buffer)}")
print(f"Object type: {type(bikeshare_stations_buffer).__name__}")
print(f"CRS: {bikeshare_stations_buffer.crs}")

In [ ]:
# Visualize the 200 metre Bike Share service areas
buffer_map = folium.Map(
    location=[43.70, -79.40],
    tiles="CartoDB positron",
    zoom_start=10
)

folium.GeoJson(
    gpd.GeoSeries(
        bikeshare_stations_buffer,
        crs="EPSG:26917"
    ).to_crs("EPSG:4326")
).add_to(buffer_map)

buffer_map

---

## Question 8c - Identify Subway Stations with Bike Share Access

Determine whether each TTC subway station lies within 200 metres of at least one Bike Share station.

Create a Boolean column named `bike_access` in `subway_stations`:

- `True` if the subway station falls within at least one 200 metre Bike Share buffer,
- `False` otherwise.

Use a spatial join rather than manually looping over stations.

In [ ]:
# Question 8c

# Convert the Bike Share station buffers to a GeoDataFrame.
bike_buffers_gdf = ...

# Spatially join subway stations to the 200 m Bike Share buffers.
subway_buffer_matches = gpd.sjoin(
    ...,
    ...,
    how="left",
    predicate="within"
)

# A subway station may match more than one Bike Share buffer.
# For each subway-station index, determine whether at least one
# matching buffer was found.
stations_with_access = (
    subway_buffer_matches
    .groupby(subway_buffer_matches.index)["index_right"]
    .apply(...)
)

# Add a Boolean "bike_access" column to subway_stations.
# Stations with no match should be False.
subway_stations["bike_access"] = ...

subway_stations.head()

In [ ]:
# Verification - do not modify
print("Q8c Answer - Bike Share access:")
print(subway_stations["bike_access"].value_counts())

print(
    "All values Boolean:",
    subway_stations["bike_access"].dtype == bool
)

In [ ]:
# Create an interactive map of subway access
subway_access_map = folium.Map(
    location=[43.70, -79.40],
    tiles="CartoDB positron",
    zoom_start=10
)

# Add Bike Share service buffers
folium.GeoJson(
    bike_buffers_gdf.to_crs("EPSG:4326"),
    style_function=lambda feature: {
        "fillOpacity": 0.1,
        "weight": 0.5
    }
).add_to(subway_access_map)

# Add TTC subway stations
for _, row in subway_stations.to_crs("EPSG:4326").iterrows():

    marker_color = "green" if row["bike_access"] else "red"

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=row["STATION"],
        icon=folium.Icon(color=marker_color)
    ).add_to(subway_access_map)

subway_access_map

---

## Question 8d - Percentage of Subway Stations with Bike Share Access

Calculate the percentage of TTC subway stations located within 200 metres of at least one Bike Share station.

Store the result in a variable named `bike_access`.

In [ ]:
# Question 8d

# Calculate the percentage of subway stations with Bike Share access.
# Hint: the mean of a Boolean Series gives the proportion of True values.
bike_access = ...

print(
    f"{bike_access:.1f}% of subway stations are within "
    "200 metres of a Bike Share station."
)

---

# Submission

Before submitting your assignment:

1. Restart the kernel and run all cells from beginning to end.
2. Make sure all code cells execute without errors.
3. Confirm that all static and interactive maps display correctly.
4. Remove unnecessary scratch cells and excessive output.
5. Save your completed notebook (`.ipynb`).
6. Export the notebook as an HTML file (`.html`).
7. Submit **both the `.ipynb` and `.html` files** to Quercus.

Make sure your final notebook contains only the code, outputs, maps, and written responses needed to answer the assignment questions.